In [3]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import MinMaxScaler

sys.path.insert(0, "src")

from sugar_jepa.sugar_jepa_model import JepaEncoder
from common.evaluation.device import resolve_torch_device

device = torch.device(resolve_torch_device())
print(f"using {device}")

DATA = Path("data/input/loop_ai_ready_joined2.csv")
JEPA_RUNS = Path("jepa_nudes")
WINDOW_STRIDE = 4
WINDOW_INPUT = 128
HORIZON = 12

using mps


In [2]:
JEPA_ENCODER_ARGS = {"n_time_steps", "patch_size", "embed_dim", "n_layers", "n_heads", "mlp_ratio", "dropout", "norm"}

def load_jepa_encoder(run_dir, checkpoint="encoder.pt"):
    run_dir = Path(run_dir)
    config = json.loads((run_dir / "config.json").read_text())
    kwargs = {k: v for k, v in config.items() if k in JEPA_ENCODER_ARGS}
    encoder = JepaEncoder(**kwargs)
    state_dict = torch.load(run_dir / checkpoint, map_location=device)
    encoder.load_state_dict(state_dict)
    encoder.requires_grad_(False)
    return encoder.to(device).eval(), config["n_time_steps"]

jepa_encoder, WINDOW_JEPA = load_jepa_encoder(
    JEPA_RUNS / "jepa_encoder-864" / "jepa_encoder_w864_p8_d96_l3_h6_20260820_002218"
)
print(f"jepa_encoder: window={WINDOW_JEPA}, embed_dim={jepa_encoder.embed_dim}")

jepa_encoder: window=864, embed_dim=96


In [4]:
DATA_COLS = [
    "sequence_id", "Timestamp", "User ID", "Study Group", "Recommended Split",
    "Glucose (mg/dL)", "Basal Rate (U/h)", "Bolus Insulin (U)", "Carbohydrates (g)",
]

df = pd.read_csv(DATA, usecols=DATA_COLS, parse_dates=["Timestamp"])
df = df.sort_values(["User ID", "Timestamp"])

df["Glucose (mg/dL)"] = df.groupby("User ID")["Glucose (mg/dL)"].transform(lambda s: s.ffill().bfill())
df["Basal Rate (U/h)"] = df.groupby("User ID")["Basal Rate (U/h)"].transform(lambda s: s.ffill().bfill())
df["Bolus Insulin (U)"] = df["Bolus Insulin (U)"].fillna(0.0)
df["Carbohydrates (g)"] = df["Carbohydrates (g)"].fillna(0.0)
df = df.dropna(subset=["Glucose (mg/dL)"])

print(f"{len(df):,} rows, {df['User ID'].nunique():,} users")
df["Recommended Split"].value_counts()

12,090,991 rows, 2,292 users


Recommended Split
train    8390218
test     1860222
val      1840551
Name: count, dtype: int64

In [5]:
class SugarJepaWindowDataset(Dataset):
    """(jepa_glucose, x, y) triples for a frozen-JEPA-encoder SugarOne variant.

    jepa_glucose: (jepa_steps,)          raw glucose, fed straight to the frozen encoder.
    x:            (input_steps, 4)       glucose/basal/bolus/carbs, MinMax-scaled.
    y:            (horizon,)             future glucose, MinMax-scaled.

    mode="disjoint": x starts right after the jepa window ends (span = jepa_steps + input_steps + horizon).
    mode="overlap":  x is the last input_steps of the jepa window itself (span = jepa_steps + horizon).
    """

    
    CHANNELS = ["Glucose (mg/dL)", "Basal Rate (U/h)", "Bolus Insulin (U)", "Carbohydrates (g)"]

    def __init__(
        self,
        df,
        jepa_steps,
        input_steps,
        horizon,
        scalers,
        mode="disjoint",
        user_col="User ID",
        stride=WINDOW_STRIDE,
    ):
        assert mode in ("disjoint", "overlap")
        self.jepa_steps = jepa_steps
        self.input_steps = input_steps
        self.horizon = horizon
        self.mode = mode

        span = jepa_steps + horizon if mode == "overlap" else jepa_steps + input_steps + horizon

        self._raw_glucose, self._scaled = [], []
        self._index = []  # (series_idx, jepa_start)

        n_skipped = 0
        for _, g in df.groupby(user_col, sort=False):
            g = g.sort_values("Timestamp")
            if len(g) < span:
                n_skipped += 1
                continue

            raw_glucose = g["Glucose (mg/dL)"].to_numpy(dtype=np.float32)
            scaled = np.stack(
                [scalers[col].transform(g[[col]].to_numpy(dtype=np.float32)).ravel() for col in self.CHANNELS],
                axis=-1,
            ).astype(np.float32)

            si = len(self._raw_glucose)
            self._raw_glucose.append(raw_glucose)
            self._scaled.append(scaled)

            n_windows = len(g) - span + 1
            for jepa_start in range(0, n_windows, stride):
                self._index.append((si, jepa_start))

        if n_skipped:
            print(f"skipped {n_skipped} users shorter than {span} steps ({mode})")

    def __len__(self):
        return len(self._index)

    def __getitem__(self, idx):
        si, jepa_start = self._index[idx]
        raw_glucose = self._raw_glucose[si]
        scaled = self._scaled[si]

        jepa = raw_glucose[jepa_start : jepa_start + self.jepa_steps]
        if self.mode == "overlap":
            now = jepa_start + self.jepa_steps
            x_start = now - self.input_steps
        else:
            x_start = jepa_start + self.jepa_steps
            now = x_start + self.input_steps

        x = scaled[x_start : x_start + self.input_steps]
        y = scaled[now : now + self.horizon, 0]  # glucose channel only

        return torch.from_numpy(jepa.copy()), torch.from_numpy(x.copy()), torch.from_numpy(y.copy())

In [6]:
train_df = df[df["Recommended Split"] == "train"]
val_df = df[df["Recommended Split"] == "val"]
test_df = df[df["Recommended Split"] == "test"]

scalers = {
    col: MinMaxScaler().fit(train_df[[col]].to_numpy(dtype=np.float32))
    for col in SugarJepaWindowDataset.CHANNELS
}

train_ds_disjoint = SugarJepaWindowDataset(train_df, WINDOW_JEPA, WINDOW_INPUT, HORIZON, scalers, mode="disjoint")
val_ds_disjoint = SugarJepaWindowDataset(val_df, WINDOW_JEPA, WINDOW_INPUT, HORIZON, scalers, mode="disjoint")

train_ds_overlap = SugarJepaWindowDataset(train_df, WINDOW_JEPA, WINDOW_INPUT, HORIZON, scalers, mode="overlap")
val_ds_overlap = SugarJepaWindowDataset(val_df, WINDOW_JEPA, WINDOW_INPUT, HORIZON, scalers, mode="overlap")

print(f"disjoint: {len(train_ds_disjoint):,} train / {len(val_ds_disjoint):,} val windows")
print(f"overlap:  {len(train_ds_overlap):,} train / {len(val_ds_overlap):,} val windows")

skipped 16 users shorter than 1004 steps (disjoint)
skipped 11 users shorter than 1004 steps (disjoint)
skipped 12 users shorter than 876 steps (overlap)
skipped 8 users shorter than 876 steps (overlap)
disjoint: 1,702,137 train / 373,066 val windows
overlap:  1,752,459 train / 384,063 val windows


In [8]:
from sugar_jepa.sugar_jepa_model import PositionalEncoding, SugarJepaParallelBlock

class SugarJepaFrozenEncoderModel(nn.Module):
    """SugarJepa with a frozen, pre-loaded JEPA encoder over an independent (longer) lookback.

    x:            (batch, input_steps, 4)  glucose/basal/bolus/carbs, MinMax-scaled.
    jepa_glucose: (batch, jepa_steps)       raw glucose, fed to the frozen encoder.
    Output: (batch, horizon)
    """

    def __init__(
        self,
        jepa_encoder,
        input_steps,
        d_model=32,
        n_heads=4,
        ff_units=128,
        n_blocks=3,
        prediction_horizon=12,
        dropout=0.1,
    ):
        super().__init__()
        self.jepa_encoder = jepa_encoder
        self.jepa_encoder.requires_grad_(False)
        self.jepa_encoder.eval()

        self.embed_glucose = nn.Linear(1, d_model)
        self.embed_basal = nn.Linear(1, d_model)
        self.embed_bolus = nn.Linear(1, d_model)
        self.embed_carbs = nn.Linear(1, d_model)

        self.pos_enc = PositionalEncoding(d_model, max_len=input_steps)
        self.jepa_proj = nn.Linear(jepa_encoder.embed_dim, d_model)

        self.blocks = nn.ModuleList(
            [
                SugarJepaParallelBlock(d_model, n_heads, ff_units, dropout, batch_first=True)
                for _ in range(n_blocks)
            ]
        )

        self.flatten_fc = nn.Linear(d_model * input_steps, d_model)
        self.out_fc = nn.Linear(d_model, prediction_horizon)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, jepa_glucose):
        g = x[..., 0:1]
        b = x[..., 1:2]
        bo = x[..., 2:3]
        c = x[..., 3:4]

        g_e = self.pos_enc(self.embed_glucose(g))
        b_e = self.pos_enc(self.embed_basal(b))
        bo_e = self.pos_enc(self.embed_bolus(bo))
        c_e = self.pos_enc(self.embed_carbs(c))

        self.jepa_encoder.eval()
        with torch.no_grad():
            jepa_e = self.jepa_encoder(jepa_glucose)  # (batch, n_patches, embed_dim)
        jepa_e = self.jepa_proj(jepa_e)               # (batch, n_patches, d_model)

        out = g_e
        for block in self.blocks:
            out = block(out, b_e, bo_e, c_e, jepa_e)

        out = out.transpose(1, 2)
        out = out.reshape(out.size(0), -1)
        out = self.dropout(F.gelu(self.flatten_fc(out)))
        return self.out_fc(out)

model = SugarJepaFrozenEncoderModel(jepa_encoder, input_steps=WINDOW_INPUT, prediction_horizon=HORIZON).to(device)
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"trainable: {n_trainable:,} / total: {n_total:,}")

trainable: 274,648 / total: 611,224


In [9]:
BATCH_SIZE = 256

train_dl_disjoint = DataLoader(train_ds_disjoint, batch_size=BATCH_SIZE, shuffle=True)
val_dl_disjoint = DataLoader(val_ds_disjoint, batch_size=BATCH_SIZE)

train_dl_overlap = DataLoader(train_ds_overlap, batch_size=BATCH_SIZE, shuffle=True)
val_dl_overlap = DataLoader(val_ds_overlap, batch_size=BATCH_SIZE)

In [10]:
import lightning as L

class LightningForecaster(L.LightningModule):
    def __init__(self, model, lr=1e-3):
        super().__init__()
        self.model = model
        self.lr = lr

    def _step(self, batch, stage):
        jepa, x, y = batch
        pred = self.model(x, jepa)
        loss = F.mse_loss(pred, y)
        mae = (pred - y).abs().mean()
        self.log(f"{stage}_loss", loss, prog_bar=True, on_epoch=True)
        self.log(f"{stage}_mae", mae, prog_bar=True, on_epoch=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._step(batch, "val")

    def configure_optimizers(self):
        trainable = filter(lambda p: p.requires_grad, self.model.parameters())
        return torch.optim.Adam(trainable, lr=self.lr)

In [ ]:
from lightning.pytorch.loggers import CSVLogger

logger_disjoint = CSVLogger(save_dir="runs/sugar_jepa_frozen_logs", name="disjoint")

model_disjoint = SugarJepaFrozenEncoderModel(jepa_encoder, input_steps=WINDOW_INPUT, prediction_horizon=HORIZON).to(device)
lit_disjoint = LightningForecaster(model_disjoint)

trainer_disjoint = L.Trainer(
    max_epochs=15,
    enable_checkpointing=False,
    logger=logger_disjoint,
)
trainer_disjoint.fit(lit_disjoint, train_dl_disjoint, val_dl_disjoint)

In [ ]:
logger_overlap = CSVLogger(save_dir="runs/sugar_jepa_frozen_logs", name="overlap")

model_overlap = SugarJepaFrozenEncoderModel(jepa_encoder, input_steps=WINDOW_INPUT, prediction_horizon=HORIZON).to(device)
lit_overlap = LightningForecaster(model_overlap)

trainer_overlap = L.Trainer(
    max_epochs=15,
    enable_checkpointing=False,
    logger=logger_overlap,
)
trainer_overlap.fit(lit_overlap, train_dl_overlap, val_dl_overlap)

In [ ]:
import matplotlib.pyplot as plt

def epoch_series(csv_path, col):
    m = pd.read_csv(csv_path)
    m = m.dropna(subset=[col])
    return m["epoch"], m[col]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for name, log_dir in [("disjoint", logger_disjoint.log_dir), ("overlap", logger_overlap.log_dir)]:
    csv_path = Path(log_dir) / "metrics.csv"

    ep, val = epoch_series(csv_path, "train_loss_epoch")
    axes[0].plot(ep, val, label=f"{name} train")
    ep, val = epoch_series(csv_path, "val_loss")
    axes[0].plot(ep, val, linestyle="--", label=f"{name} val")

    ep, val = epoch_series(csv_path, "train_mae_epoch")
    axes[1].plot(ep, val, label=f"{name} train")
    ep, val = epoch_series(csv_path, "val_mae")
    axes[1].plot(ep, val, linestyle="--", label=f"{name} val")

axes[0].set_title("MSE Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].set_title("MAE (scaled [0,1] space)"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
plt.show()